# Appendix C.6 — Sustainable Farming Incentive

Evidence for the claims made about the SFI datasets in Appendix C.

**Sources used**

| file | what it is |
| --- | --- |
| `raw_datasets/poole_harbour_rivers_sustainable_farming_initiatives.geojson` | the drawn SFI actions in the catchment |
| `raw_datasets/SFI Option details.xlsx` | the option/payment workbook (expanded offer) |
| `raw_datasets/Scheme details.xlsx` | the FARMSCOPER pollutant-impact sheet |


In [1]:
import os, json, warnings
from pathlib import Path
import pandas as pd

warnings.filterwarnings("ignore")
ROOT = Path.cwd()
while not (ROOT / "raw_datasets").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
os.chdir(ROOT)
RAW = ROOT / "raw_datasets"
REG = RAW / "access_database_csv_files"

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 60)
pd.set_option("display.max_colwidth", 90)
print("repository root:", ROOT)


repository root: /Users/waf/git/projects/demonstrator-poc


In [2]:
import geopandas as gpd

sfi = gpd.read_file(RAW / "poole_harbour_rivers_sustainable_farming_initiatives.geojson")
options = pd.read_excel(RAW / "SFI Option details.xlsx", sheet_name="SFI Option Data")
farmscoper = pd.read_excel(RAW / "Scheme details.xlsx", sheet_name="Farmscoper")
print(f"drawn SFI action rows in the catchment: {len(sfi):,}")
print(f"options described in the workbook      : {len(options):,}")
print(f"FARMSCOPER treatments                  : {len(farmscoper):,}")
sfi.drop(columns="geometry").head(3)


drawn SFI action rows in the catchment: 35,301
options described in the workbook      : 102
FARMSCOPER treatments                  : 106


,id,app_id,ref_year,contract_start,contract_end,scheme,application_type,option_code,area,mtl,units,uom_desc,opt_year,schememodule
0,Sustainable_Farming_Incentive_Scheme_Actions_England.641449,1707990,2024,2024-03-01 00:00:00+00:00,2027-02-28 00:00:00+00:00,SFI 23,SFI Actions,HRW2,NaN,698.0,NaN,Metres,2026,SFI23
1,Sustainable_Farming_Incentive_Scheme_Actions_England.641451,1707990,2024,2024-03-01 00:00:00+00:00,2027-02-28 00:00:00+00:00,SFI 23,SFI Actions,HRW1,NaN,698.0,NaN,Metres,2026,SFI23
2,Sustainable_Farming_Incentive_Scheme_Actions_England.641524,1689263,2024,2024-04-01 00:00:00+00:00,2027-03-31 00:00:00+00:00,SFI 23,SFI Actions,SAM1,3.6071,NaN,NaN,HA,2026,SFI23


---
## C.6.1 Only part of the offer is priced

> *"Only Expanded Offer rates are published in the source used, leaving SFI 2023 agreements unpriced."*


In [3]:
priced = set(options.Code.dropna().astype(str))
codes = sfi.option_code.dropna().astype(str)
unpriced_codes = sorted(set(codes) - priced)
unpriced_rows = sfi[sfi.option_code.isin(unpriced_codes)]

print(f"distinct option codes drawn in the catchment : {codes.nunique()}")
print(f"of those, absent from the payment workbook   : {len(unpriced_codes)}")
print(f"drawn rows they account for                  : {len(unpriced_rows):,} of {len(sfi):,} "
      f"({100*len(unpriced_rows)/len(sfi):.0f}%)\n")
print("by scheme:")
print(unpriced_rows.scheme.value_counts().to_string())


distinct option codes drawn in the catchment : 129
of those, absent from the payment workbook   : 58
drawn rows they account for                  : 20,032 of 35,301 (57%)

by scheme:
scheme
SFI 23    19226
SFI EO      806


In [4]:
print("The unpriced codes:")
print(", ".join(unpriced_codes))
print("\nNote the C-prefixed ones -- CAB10, CGS16, CWT2 and the rest are EXPANDED OFFER codes.")
print("The workbook does not even cover the whole expanded offer:")
eo_unpriced = unpriced_rows[unpriced_rows.scheme == "SFI EO"]
print(f"  expanded-offer rows with no payment rate: {len(eo_unpriced):,}")
print(f"  distinct expanded-offer codes affected  : {eo_unpriced.option_code.nunique()}")


The unpriced codes:
AHL1, AHL2, AHL3, AHL4, CAB10, CAB14, CAB16, CAB18, CAGF1, CBE4, CGS16, CGS18, CGS19, CGS21, CGS22, CGS23, CGS26, CGS4, CHRW4, CPAC1, CSP1, CSP11, CSP16, CSP17, CSP21, CSP6, CSP9, CSW22, CSW25, CWD2, CWD20, CWD7, CWD8, CWS1, CWS10, CWS11, CWS3, CWS8, CWT12, CWT13, CWT15, CWT2, CWT3, HRW1, HRW2, HRW3, IGL1, IGL2, IGL3, IPM2, IPM3, IPM4, LIG1, NUM2, NUM3, SAM1, SAM2, SAM3

Note the C-prefixed ones -- CAB10, CGS16, CWT2 and the rest are EXPANDED OFFER codes.
The workbook does not even cover the whole expanded offer:
  expanded-offer rows with no payment rate: 806
  distinct expanded-offer codes affected  : 39


In [5]:
print("Example agreements that therefore cannot be costed at all:")
sfi[sfi.option_code.isin(unpriced_codes)][
    ["app_id","scheme","option_code","area","mtl","uom_desc","contract_start","contract_end"]].head(8)


Example agreements that therefore cannot be costed at all:


,app_id,scheme,option_code,area,mtl,uom_desc,contract_start,contract_end
0,1707990,SFI 23,HRW2,NaN,698.0,Metres,2024-03-01 00:00:00+00:00,2027-02-28 00:00:00+00:00
1,1707990,SFI 23,HRW1,NaN,698.0,Metres,2024-03-01 00:00:00+00:00,2027-02-28 00:00:00+00:00
2,1689263,SFI 23,SAM1,3.6071,NaN,HA,2024-04-01 00:00:00+00:00,2027-03-31 00:00:00+00:00
3,1689263,SFI 23,SAM2,3.5000,NaN,HA,2024-04-01 00:00:00+00:00,2027-03-31 00:00:00+00:00
4,1689263,SFI 23,IPM4,3.6071,NaN,HA,2024-04-01 00:00:00+00:00,2027-03-31 00:00:00+00:00
5,1787503,SFI 23,SAM1,2.4464,NaN,HA,2024-08-01 00:00:00+00:00,2027-07-31 00:00:00+00:00
122,1723211,SFI 23,SAM1,2.6196,NaN,HA,2024-05-01 00:00:00+00:00,2027-04-30 00:00:00+00:00
123,1723211,SFI 23,HRW3,NaN,170.0,Metres,2024-05-01 00:00:00+00:00,2027-04-30 00:00:00+00:00


---
## C.6.2 Payment rates are only partly machine-actionable

> *"Per-hectare and per-100-metre rates can be applied to a mapped extent; per square metre, per plot,
> per tonne, per assessment and the multi-clause hectarage-recipe variants cannot."*


In [6]:
units = options["Pay Unit"].fillna("(none)").astype(str).str.strip()
simple = units.isin(["per hectare", "per 100 metres"])
print(f"options whose pay unit is a clean, applicable quantity: {simple.sum()} of {len(options)}")
print(f"options whose pay unit is prose, or absent            : {(~simple).sum()}\n")
pd.DataFrame({"pay unit": units[~simple].values,
              "code": options.Code[~simple].values,
              "payment": options.Payment[~simple].values}).reset_index(drop=True)


options whose pay unit is a clean, applicable quantity: 87 of 102
options whose pay unit is prose, or absent            : 15



,pay unit,code,payment
0,(none),WBD1,NaN
1,per tonne per year,AHW2,732.0
2,per square metre,HEF1,5.0
3,assessment and review report per year,CNUM1,652.0
4,assessment and plan per year,CIPM1,1129.0
5,(none),WBD2,NaN
6,per hectare calculate the hectarage by: measuring the length of the buffer strip in me...,BFS1,707.0
7,per hectare calculate the hectarage by: measuring the length of the buffer strip in me...,CIGL3,235.0
8,per hectare you can only include the land area where you€™ll do this action (not the a...,HEF8,2512.0
9,per plot per year (minimum 2 plots per hectare (ha)),AHW4,11.0


Several of those "units" are calculation instructions written for a human — *"measuring the length of
the buffer strip in metres, multiplying that length by the relevant width (10m to 20m)…"*. The width is
a range, so even a human cannot produce one number from it.

And the conditions that change what is payable live in a free-text notes column:


In [7]:
notes = options[options.More_pay_info.notna()][["Code","Payment","Pay Unit","More_pay_info"]]
print(f"{len(notes)} options carry qualifying payment text:\n")
notes.head(10)


13 options carry qualifying payment text:



,Code,Payment,Pay Unit,More_pay_info
3,CSAM1,6.0,per hectare,and £97 per SFI agreement per year
7,CHRW2,13.0,per 100 metres,for one side of an eligible hedgerow per year
11,CHRW1,5.0,per 100 metres,for one side of an eligible hedgerow per year
14,CHRW3,10.0,per 100 metres,(m) for both sides of an eligible hedgerow per year
19,AHW2,732.0,per tonne per year,maximum of 1 tonne of supplementary winter bird food (action AHW2) for every 2 hectare...
20,BFS6,742.0,per hectare,€“ calculate the hectarage by: measuring the length of the strip in metres (m) multipl...
24,HEF1,5.0,per square metre,€“ you must only include the area of the building€™s ground floor
25,AGF1,248.0,per hectare,the hectarage can include: the area used to grow agroforestry trees the areas between ...
28,AGF2,385.0,per hectare,the hectarage can include: the area used to grow agroforestry trees the areas between...
41,CAHL4,515.0,per hectare,calculate the hectarage by: measuring the length of the buffer strip in metres (m) mu...


In [8]:
print("Two that change the answer, not the wording:")
for code in ["CHRW3", "CSAM1"]:
    row = options[options.Code == code].iloc[0]
    print(f"  {code}: £{row.Payment} {row['Pay Unit']} -- {row.More_pay_info}")
print()
print("CHRW3 pays for BOTH sides of a hedgerow: extent x rate under-counts unless the mapped length")
print("is already doubled -- and nothing says whether it is.")
print("CSAM1 adds a flat £97 per agreement, which no per-hectare arithmetic will ever produce.")


Two that change the answer, not the wording:
  CHRW3: £10.0 per 100 metres  -- (m) for both sides of an eligible hedgerow per year
  CSAM1: £6.0 per hectare --  and £97 per SFI agreement per year

CHRW3 pays for BOTH sides of a hedgerow: extent x rate under-counts unless the mapped length
is already doubled -- and nothing says whether it is.
CSAM1 adds a flat £97 per agreement, which no per-hectare arithmetic will ever produce.


In [9]:
print("Duration is free text too, so an agreement's term is not machine-readable either:")
options[options.duration.astype(str).str.len() > 12][["Code","duration"]].head(5)


Duration is free text too, so an agreement's term is not machine-readable either:


,Code,duration
68,OFC2,1 year you can apply for this action for a maximum of 2 consecutive years to cover the...
79,OFC1,1 year you can apply for this action for a maximum of 2 consecutive years to cover the...
81,OFC3,1 year you can apply for this action for a maximum of 2 consecutive years to cover the...
88,OFC5,1 year you can apply for this action for a maximum of 3 consecutive years to cover the...
98,OFC4,1 year you can apply for this action for a maximum of 2 consecutive years to cover the...


In [10]:
# The workbook also carries mojibake, which propagates into any label taken from it verbatim.
text_cols = options.select_dtypes(include="object")
mask = text_cols.apply(lambda c: c.astype(str).str.contains("€", regex=True, na=False))
print(f"cells containing a stray euro sign (UTF-8 read as Windows-1252): {int(mask.values.sum())}")
print(f"option rows affected: {int(mask.any(axis=1).sum())} of {len(options)}\n")
for col in mask.columns[mask.any()][:4]:
    v = options.loc[mask[col], col].iloc[0]
    print(f"  [{col}] {str(v)[:120]}")


cells containing a stray euro sign (UTF-8 read as Windows-1252): 659
option rows affected: 102 of 102

  [Pay Unit] per hectare you can only include the land area where you€™ll do this action (not the area of water) You can calculate th
  [More_pay_info] €“ calculate the hectarage by: measuring the length of the strip in metres (m) multiplying that length by the relevant w
  [Aim] This action€™s aim is that there€™s grassland which produces a sward with: flowering grasses and wildflowers from late s
  [Aim.1] there€™s grassland which produces a sward with: flowering grasses and wildflowers from late spring and during the summer


---
## C.6.3 Pollutant removal is modelled, and the modelled unit is undocumented

> *"The headline question is what `Kg … Ha-1 Yr-1` is per hectare *of*."*


In [11]:
print("The Farmscoper sheet's columns -- this is the entire documentation of its units:")
print(list(farmscoper.columns))
print()
valued = farmscoper.dropna(subset=["Kg P Ha-1 Yr-1"])
valued = valued[valued["Kg P Ha-1 Yr-1"] != 0]
print(f"treatments carrying quantified values: {len(valued)} of {len(farmscoper)}")
valued[["Name","Kg Nitrate Ha-1 Yr-1","Kg P Ha-1 Yr-1","Kg Z Ha-1 Yr-1"]].head(8)


The Farmscoper sheet's columns -- this is the entire documentation of its units:
['Name', 'Kg Nitrate Ha-1 Yr-1', 'Kg P Ha-1 Yr-1', 'Kg Z Ha-1 Yr-1', 'OFDB Actions', 'Scheme Actions']

treatments carrying quantified values: 13 of 106


,Name,Kg Nitrate Ha-1 Yr-1,Kg P Ha-1 Yr-1,Kg Z Ha-1 Yr-1
3,Adopt reduced cultivation systems,-4.834525,-0.221994,-240.894627
9,Beetle banks,-0.625477,-0.245566,-259.281784
15,Construct troughs with concrete base,-1.007347,-0.051967,-49.641342
28,Early harvesting and establishment of crops in the autumn,-11.688292,-0.411477,-343.591220
30,Establish cover crops in the autumn,-24.491537,-0.834522,-700.601803
31,Establish in-field grass buffer strips,-1.277450,-1.520251,-1567.272604
33,Establish riparian buffer strips,-0.511700,-0.112118,-110.929369
61,Management of arable field corners,-3.504652,-1.552300,-1651.380245


A column header, a number, and a sign convention nobody states. The three open questions in Appendix C
— per hectare of the treated land or of the whole modelled farm; loss at the field edge or load delivered
to water; a real annualisation or a multi-year run divided through — are all unanswerable from the
workbook. The values are load-bearing: every applied impact figure is `extent × rate`.


In [12]:
print("Nothing in the workbook's other sheets defines them either:")
for sheet in pd.ExcelFile(RAW / "Scheme details.xlsx").sheet_names:
    df = pd.read_excel(RAW / "Scheme details.xlsx", sheet_name=sheet, header=None, nrows=40)
    hits = df.astype(str).apply(lambda c: c.str.contains("Ha-1|per hectare|denominator", case=False, na=False)).any().any()
    print(f"   {sheet:<14} mentions a per-hectare definition: {hits}")


Nothing in the workbook's other sheets defines them either:
   Farmscoper     mentions a per-hectare definition: True
   QEIA           mentions a per-hectare definition: False


   Land Use       mentions a per-hectare definition: False


   Info           mentions a per-hectare definition: False


   OFDB Actions   mentions a per-hectare definition: False


   SFI Codes      mentions a per-hectare definition: False


### The column that is not zinc

> *"`Kg Z Ha-1 Yr-1` reads as zinc, but its magnitudes reach −1,651 kg/ha/yr … and it tracks the
> phosphorus column at a near-constant ~870:1."*


In [13]:
v = valued.copy()
v["Z : P ratio"] = v["Kg Z Ha-1 Yr-1"] / v["Kg P Ha-1 Yr-1"]
print(f"Z:P ratio across all {len(v)} valued treatments -- "
      f"min {v['Z : P ratio'].min():.0f}, max {v['Z : P ratio'].max():.0f}, "
      f"mean {v['Z : P ratio'].mean():.0f}\n")
v[["Name","Kg P Ha-1 Yr-1","Kg Z Ha-1 Yr-1","Z : P ratio"]].sort_values("Kg Z Ha-1 Yr-1")


Z:P ratio across all 13 valued treatments -- min 529, max 1085, mean 880



,Name,Kg P Ha-1 Yr-1,Kg Z Ha-1 Yr-1,Z : P ratio
61,Management of arable field corners,-1.552300,-1651.380245,1063.828017
31,Establish in-field grass buffer strips,-1.520251,-1567.272604,1030.930040
72,Plant areas of farm with wild bird seed / nectar flower mixtures,-1.601561,-1491.421037,931.229794
30,Establish cover crops in the autumn,-0.834522,-700.601803,839.525111
88,Undersown spring cereals,-0.576482,-474.438734,822.990132
28,Early harvesting and establishment of crops in the autumn,-0.411477,-343.591220,835.018629
9,Beetle banks,-0.245566,-259.281784,1055.853643
3,Adopt reduced cultivation systems,-0.221994,-240.894627,1085.139302
33,Establish riparian buffer strips,-0.112118,-110.929369,989.394361
79,Reduce the length of the grazing day/grazing season,-0.104706,-74.675011,713.190226


Three independent reasons this is not zinc:

- **Magnitude.** The largest rate is −1,651 kg/ha/yr. A hectare of topsoil holds on the order of
  150–250 kg of zinc *in total*, so the action would remove several times the entire stock every year.
- **Ratio.** A near-constant ~870:1 against phosphorus is the sediment-to-particulate-P relationship
  (soil P is roughly 0.1% of sediment by mass). Zinc, at 60–100 mg/kg of soil, would give ~0.06:1 —
  four orders of magnitude out.
- **Provenance.** FARMSCOPER's pollutant set is nitrate, phosphorus, **sediment**, ammonia, nitrous
  oxide, methane, pesticides and FIOs. It does not model zinc.

Almost certainly sediment under a wrong header — but "almost certainly" is not a basis for asserting a
substance, so the demonstrator omits the column rather than publishing it as zinc.


---
## C.6.4 Vocabulary does not align with the water-quality side

> *"The nitrogen column is headed `Kg Nitrate`, while the monitored determinand is 9686 'Nitrogen, Total
> as N' — a part compared against a whole."*


In [14]:
codelist = json.loads((RAW / "determinand_codelist.json").read_text())
items = codelist["items"] if isinstance(codelist, dict) and "items" in codelist else codelist
flat = pd.json_normalize(items)
label_col = next(c for c in flat.columns if "label" in c.lower())
print("Determinands in the EA codelist whose label mentions nitrate or total nitrogen:")
hits = flat[flat[label_col].astype(str).str.contains("nitrate|nitrogen, total", case=False, na=False)]
note_col = next(c for c in flat.columns if "notation" in c.lower())
print(hits[[note_col, label_col]].head(12).to_string(index=False))
print()
print("The FARMSCOPER column is headed:", [c for c in farmscoper.columns if "Nitrate" in c][0])
print("The determinand the catchment monitors is 9686, 'Nitrogen, Total as N'.")
print("A nitrate loss compared against a total-nitrogen observation compares a part to a whole.")


Determinands in the EA codelist whose label mentions nitrate or total nitrogen:
notation                                              prefLabel
    0116                          Nitrogen, Total Oxidised as N
    0117                                           Nitrate as N
    0120                 Nitrogen, Total Oxidised : Dry Wt as N
    3508                   Nitrogen, Total Oxidised : Load as N
    3683               Nitrogen, Total Inorganic : (Calculated)
    5280                                Nitrate, Leachable as N
    5507                  Nitrate (2M KCl extractable) : Dry Wt
    5509 Nitrogen, Total Oxidised (2M KCl extractable) : Dry Wt
    5567                               Nitrate, Filtered as NO3
    5982        Pentaerythritol tetranitrate : Dry Wt :- {PETN}
    5983      Pentaerythritol tetranitrate, Leachable :- {PETN}
    5984                 Pentaerythritol tetranitrate :- {PETN}

The FARMSCOPER column is headed: Kg Nitrate Ha-1 Yr-1
The determinand the catchment mon

---
## C.6.5 A distinct land footprint is not recoverable

> *"the parcel data records different areas for different actions on the same point, and 73.7% of the
> drawn points carry more than one action."*

The source draws **points**, not field polygons. That is the finding the last cell of this section
lands on: there is no land geometry for an area to belong to, so an area is only ever a property of an
action.


In [15]:
sfi["wkt"] = sfi.geometry.apply(lambda g: g.wkt)
per_point = sfi.groupby("wkt").option_code.nunique()
print(f"distinct drawn points in the catchment      : {len(per_point):,}")
print(f"points carrying more than one action        : {100 * (per_point > 1).mean():.0f}%\n")
print("actions per drawn point:")
print(per_point.value_counts().sort_index().rename("points").to_string())


distinct drawn points in the catchment      : 12,504
points carrying more than one action        : 74%

actions per drawn point:
option_code
1     3288
2     2965
3     2154
4     1869
5     1580
6      434
7      125
8       61
9       17
10       6
11       3
12       2


In [16]:
worst = per_point.sort_values(ascending=False).index[0]
print("One drawn point, and everything claimed on it:")
sfi[sfi.wkt == worst][["app_id","scheme","option_code","area","mtl","units","uom_desc"]].reset_index(drop=True)


One drawn point, and everything claimed on it:


,app_id,scheme,option_code,area,mtl,units,uom_desc
0,1722119,SFI 23,HRW2,NaN,322.0,NaN,Metres
1,1722119,SFI 23,IPM4,12.0426,NaN,NaN,HA
2,1722119,SFI 23,HRW3,NaN,189.0,NaN,Metres
3,1722119,SFI 23,SAM2,12.0000,NaN,NaN,HA
4,1722119,SFI 23,AHL3,0.2200,NaN,NaN,HA
5,1722119,SFI 23,HRW1,NaN,511.0,NaN,Metres
6,1722119,SFI 23,AHL4,0.2900,NaN,NaN,HA
7,1722119,SFI 23,SAM1,12.5526,NaN,NaN,HA
8,2006902,SFI EO,PRF1,12.0400,NaN,NaN,HA
9,2006902,SFI EO,WBD2,NaN,1270.0,NaN,Metres


One point, two applications, twelve actions — and the areas disagree: 12.0426 ha, 12.00 ha, 12.5526 ha,
0.22 ha, 0.29 ha, all for the same piece of ground. They are areas *under an action*, not areas of land,
and they are not intended to be summed.

Sum them anyway — which is what "total area under improvement" means to a reader — and you double-count
most of the catchment:


In [17]:
ha = sfi[sfi.uom_desc == "HA"]
naive_total = ha["area"].sum()                      # the ATTRIBUTE, not GeoSeries.area
per_point_max = ha.groupby("wkt")["area"].max().sum()
print(f"sum of every action's area          : {naive_total:>12,.0f} ha")
print(f"largest single action per drawn point: {per_point_max:>12,.0f} ha")
catchment_ha = gpd.read_file(RAW / "poole_harbour_rivers_operational_catchment.geojson").to_crs(27700).area.sum() / 10_000
print(f"the operational catchment itself    : {catchment_ha:>12,.0f} ha")
print("\nThe first number is not a land area. Neither is the second -- it is a lower bound. The")
print("source does not carry a field geometry, so the real footprint is not derivable at all.")


sum of every action's area          :      125,302 ha
largest single action per drawn point:       66,969 ha


the operational catchment itself    :       73,838 ha

The first number is not a land area. Neither is the second -- it is a lower bound. The
source does not carry a field geometry, so the real footprint is not derivable at all.
